# Decidir contra alguien: minimax y poda alfa-beta

**Facsímil 2 · Inteligencia clásica** — capítulo 11 (juegos: decidir con otros actores).

Hasta ahora has buscado en mundos «pasivos»: un mapa, un sudoku, unos bloques que no te llevan la
contraria. Pero muchísimas decisiones reales se toman **contra alguien** que responde: un rival, un
mercado, otro agente. La teoría que gobierna eso es la **teoría de juegos**, y su algoritmo
fundamental es **minimax**. En este cuaderno construyes un jugador de tres en raya que **no pierde
nunca** —no por trucos, sino porque calcula todas las jugadas suyas y del rival hasta el final,
suponiendo que el otro juega lo mejor posible—. Luego lo haces eficiente de tres maneras
(guardar posiciones repetidas, **poda alfa-beta** y **ordenar** las jugadas), mides cuánto trabajo
ahorra cada una sin cambiar ni una decisión, y terminas viendo qué pasa cuando el árbol es tan grande
que hay que **cortar y estimar**, como en el ajedrez.

### Qué vas a aprender
- Qué es un **juego de suma cero con información perfecta** y por qué se puede resolver «pensando hasta
  el final».
- Cómo funciona **minimax**, y qué es el **valor de un juego** (el del tres en raya es 0: empate con
  juego perfecto).
- Por qué la misma posición se calcula miles de veces y cómo una **tabla de transposición** (memoización)
  lo arregla.
- Cómo la **poda alfa-beta** llega a la misma decisión mirando una fracción del árbol, y por qué el
  **orden** en que exploras cambia cuánto poda.
- Qué pasa al **cortar a una profundidad** y **estimar** con una función de evaluación: el puente hacia
  el ajedrez y, más allá, hacia el aprendizaje por refuerzo.

### Cuánto cuesta
Unos 15 minutos. CPU, solo Python estándar (con un par de gráficos de matplotlib).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import math, random
import matplotlib.pyplot as plt

GANAN = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
def ganador(t):
    for a,b,c in GANAN:
        if t[a] and t[a]==t[b]==t[c]: return t[a]
    return None
def lleno(t): return all(t)
def pinta(t):
    s = [c if c else str(i+1) for i,c in enumerate(t)]
    print("\n".join(" ".join(s[i:i+3]) for i in (0,3,6)))
print("Tablero vacio (los numeros marcan las casillas libres):")
pinta([""]*9)


## 1. Qué tipo de juego es el tres en raya

El tres en raya tiene tres propiedades que lo hacen «resoluble» por completo:

- **Suma cero:** lo que uno gana, el otro lo pierde. No hay cooperación posible.
- **Información perfecta:** ambos ven el tablero entero; no hay cartas ocultas ni azar.
- **Finito:** las partidas terminan en a lo sumo 9 jugadas.

Cuando se cumplen estas tres, existe una estrategia óptima que se puede *calcular* explorando el árbol
completo de partidas. La pieza que lo hace es minimax.


## 2. Minimax: pensar hasta el final asumiendo lo peor del rival

`X` quiere ganar (**maximiza**), `O` quiere que `X` pierda (**minimiza**). El algoritmo explora el
árbol de todas las partidas posibles. En las hojas (partidas terminadas) asigna un valor: `+1` si gana
`X`, `-1` si gana `O`, `0` si hay empate. Y hacia arriba: en un turno de `X` se queda con el **máximo**
de los hijos (la mejor jugada para `X`); en un turno de `O`, con el **mínimo** (la peor para `X`,
porque `O` juega óptimo). Así, cada estado recibe el valor que tendría *si ambos jugaran perfecto* a
partir de ahí. Contamos cuántos estados visita.


In [ ]:
nodos = [0]
def minimax(t, jugador):
    nodos[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if jugador == "X":
        return max(minimax(t[:i]+["X"]+t[i+1:], "O") for i in libres)
    else:
        return min(minimax(t[:i]+["O"]+t[i+1:], "X") for i in libres)

nodos[0] = 0
valor = minimax([""]*9, "X")
print(f"Valor del juego desde cero: {valor}  (0 = con juego perfecto, siempre empate)")
print(f"Estados visitados por minimax puro: {nodos[0]:,}")


**Lo que dice ese 0.** El tres en raya, jugado perfecto por ambos, **siempre acaba en empate**:
nadie puede forzar una victoria. Minimax no «cree» que ganará; calcula que lo mejor alcanzable contra
un rival perfecto es no perder. Y para saberlo ha tenido que mirar más de medio millón de estados. Ahí
empieza el problema: ¿de verdad hacen falta tantos?


## 3. La misma posición, una y otra vez: tabla de transposición

Fíjate en algo: a la posición «`X` en el centro y `O` en una esquina» se llega de dos formas (primero
el centro y luego la esquina, o al revés). Son **caminos distintos** que desembocan en el **mismo
tablero**. Minimax puro no se da cuenta: vuelve a calcular ese tablero entero cada vez que lo
encuentra. La solución es de manual: **guardar en un diccionario** el valor de cada posición ya
resuelta (su clave es el tablero). Eso es una **tabla de transposición** (memoización). Medimos
cuántas posiciones **distintas** existen de verdad frente al medio millón de visitas.


In [ ]:
memo = {}
visitas = [0]
def minimax_memo(t, jugador):   # t es una TUPLA (sirve de clave)
    visitas[0] += 1
    clave = (t, jugador)
    if clave in memo: return memo[clave]     # ya resuelta: no recalcular
    g = ganador(t)
    if g == "X": memo[clave] = 1; return 1
    if g == "O": memo[clave] = -1; return -1
    if lleno(t): memo[clave] = 0; return 0
    libres = [i for i in range(9) if not t[i]]
    sig = "O" if jugador == "X" else "X"
    hijos = [minimax_memo(t[:i]+(jugador,)+t[i+1:], sig) for i in libres]
    v = max(hijos) if jugador == "X" else min(hijos)
    memo[clave] = v
    return v

memo.clear(); visitas[0] = 0
valor_m = minimax_memo(tuple([""]*9), "X")
print(f"Valor del juego (con memoria): {valor_m}  (igual que antes: 0)")
print(f"Posiciones UNICAS que existen de verdad: {len(memo):,}")
print(f"Estados que visitaba minimax puro:        {nodos[0]:,}")
print(f"Cada posicion unica se recalculaba de media {nodos[0]/len(memo):.0f} veces.")


Ese es el primer ahorro y la primera idea reutilizable: **no recalcular lo ya resuelto**. La tabla de
transposición es exactamente lo que usan los motores de ajedrez modernos (allí las posiciones repetidas
se cuentan por millones). A partir de aquí, nuestro **jugador perfecto** consultará esta tabla, así que
decidir una jugada será instantáneo.


In [ ]:
def mejor_jugada(t, jugador):
    # decide la jugada optima consultando la tabla de transposicion (juego perfecto)
    sig = "O" if jugador == "X" else "X"
    libres = [i for i in range(9) if not t[i]]
    punt = [(minimax_memo(tuple(t[:i]+[jugador]+t[i+1:]), sig), i) for i in libres]
    return (max if jugador == "X" else min)(punt)[1]

ej = ["X","O","X","","O","","","",""]
print("Posicion de ejemplo:"); pinta(ej)
print(f"\nLa jugada optima de X es la casilla {mejor_jugada(ej, 'X')+1}.")


## 4. Poda alfa-beta: dejar de mirar lo que ya no puede importar

La memoización evita repetir; la **poda alfa-beta** evita *mirar de más*. La intuición es de sentido
común. Imagina que evalúas una jugada y, a mitad, descubres que el rival tiene una respuesta que la
hace **peor** que otra jugada que ya tenías guardada. ¿Para qué seguir? La vas a descartar igual. La
poda formaliza esto con dos cotas, **α** (lo mejor que `X` tiene asegurado) y **β** (lo mejor que `O`
tiene asegurado): cuando se cruzan ($\beta \le \alpha$), se **corta** la rama. Da **exactamente la
misma decisión** que minimax mirando una fracción.


In [ ]:
nodos_ab = [0]
def alfabeta(t, jugador, alfa, beta):
    nodos_ab[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if jugador == "X":
        valor = -math.inf
        for i in libres:
            valor = max(valor, alfabeta(t[:i]+["X"]+t[i+1:], "O", alfa, beta))
            alfa = max(alfa, valor)
            if beta <= alfa: break        # poda: O no permitira que lleguemos aqui
        return valor
    else:
        valor = math.inf
        for i in libres:
            valor = min(valor, alfabeta(t[:i]+["O"]+t[i+1:], "X", alfa, beta))
            beta = min(beta, valor)
            if beta <= alfa: break
        return valor

nodos_ab[0] = 0
valor_ab = alfabeta([""]*9, "X", -math.inf, math.inf)
print(f"Misma decision (valor {valor_ab}), pero estados visitados: {nodos_ab[0]:,}")
print(f"\nMinimax puro:   {nodos[0]:>7,} estados")
print(f"Con poda a-b:   {nodos_ab[0]:>7,} estados")
print(f"Ahorro: {100*(1-nodos_ab[0]/nodos[0]):.0f}% del arbol, sin cambiar ni una jugada.")


## 5. El orden importa: la poda premia mirar primero lo bueno

La poda es más potente cuanto **antes** encuentres las buenas jugadas, porque así las cotas α y β se
estrechan pronto y cortan más ramas. Lo medimos con tres formas de ordenar las casillas libres: el
**orden natural** (0, 1, 2...), un **orden dirigido** (probar primero el centro, luego las esquinas y
por último los lados, que es donde suele estar lo bueno) y un **orden aleatorio**.


In [ ]:
PRIORIDAD = [4,0,2,6,8,1,3,5,7]   # centro, esquinas, lados: de mejor a peor
def alfabeta_modo(t, jugador, alfa, beta, cont, modo):
    cont[0] += 1
    g = ganador(t)
    if g == "X": return 1
    if g == "O": return -1
    if lleno(t): return 0
    libres = [i for i in range(9) if not t[i]]
    if modo == "dirigido":   libres.sort(key=lambda i: PRIORIDAD.index(i))
    elif modo == "aleatorio": random.shuffle(libres)
    if jugador == "X":
        valor = -math.inf
        for i in libres:
            valor = max(valor, alfabeta_modo(t[:i]+["X"]+t[i+1:], "O", alfa, beta, cont, modo))
            alfa = max(alfa, valor)
            if beta <= alfa: break
        return valor
    else:
        valor = math.inf
        for i in libres:
            valor = min(valor, alfabeta_modo(t[:i]+["O"]+t[i+1:], "X", alfa, beta, cont, modo))
            beta = min(beta, valor)
            if beta <= alfa: break
        return valor

cnat = [0]; alfabeta_modo([""]*9, "X", -math.inf, math.inf, cnat, "natural")
cdir = [0]; alfabeta_modo([""]*9, "X", -math.inf, math.inf, cdir, "dirigido")
random.seed(0)
muestras = []
for _ in range(20):
    c = [0]; alfabeta_modo([""]*9, "X", -math.inf, math.inf, c, "aleatorio"); muestras.append(c[0])
media_alea = sum(muestras)//len(muestras)
print(f"Sin poda (minimax):          {nodos[0]:>7,} estados")
print(f"Poda, orden ALEATORIO:       {min(muestras):>7,} a {max(muestras):,} (media {media_alea:,})")
print(f"Poda, orden NATURAL:         {cnat[0]:>7,} estados")
print(f"Poda, orden DIRIGIDO:        {cdir[0]:>7,} estados")
print("\nLa poda siempre ayuda; CUANTO ayuda depende del orden. Mirar primero lo bueno poda mas.")


## 6. Las cuatro estrategias, de un vistazo

Juntamos los números en un gráfico (escala logarítmica, porque van de unos pocos miles a más de medio
millón). De izquierda a derecha: minimax puro, las posiciones únicas que existen de verdad, y la poda
con orden natural y con orden dirigido. El mensaje es que **la misma decisión** (siempre el empate) se
puede alcanzar con esfuerzos que se diferencian en dos órdenes de magnitud.


In [ ]:
etiquetas = ["minimax\npuro", "posiciones\nunicas", "alfa-beta\nnatural", "alfa-beta\ndirigido"]
valores = [nodos[0], len(memo), cnat[0], cdir[0]]
fig, ax = plt.subplots(figsize=(6.2,3.6))
barras = ax.bar(etiquetas, valores, color=["#555","#888","#4c72b0","#2a9d4a"])
ax.set_yscale("log"); ax.set_ylabel("estados (escala log)")
ax.set_title("Mismo resultado, muy distinto esfuerzo")
for b, v in zip(barras, valores):
    ax.text(b.get_x()+b.get_width()/2, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()


## 7. ¿De verdad no pierde? Que juegue 2000 partidas contra el azar

La prueba definitiva: ponemos a nuestro `X` (que decide con la tabla de transposición, es decir, juego
perfecto) a jugar contra un `O` que mueve al azar. Como `X` juega perfecto, **no debería perder ni una
sola** vez.


In [ ]:
def azar(t, jugador):
    return random.choice([i for i in range(9) if not t[i]])

def juega(jug_X, jug_O, n, semilla):
    random.seed(semilla)
    res = {"gana X": 0, "empate": 0, "gana O": 0}
    for _ in range(n):
        t = [""]*9; turno = "X"
        while ganador(t) is None and not lleno(t):
            i = jug_X(t, "X") if turno == "X" else jug_O(t, "O")
            t[i] = turno; turno = "O" if turno == "X" else "X"
        g = ganador(t)
        res["gana X" if g=="X" else "gana O" if g=="O" else "empate"] += 1
    return res

res = juega(mejor_jugada, azar, 2000, semilla=1)
print("Resultado de 2000 partidas (nuestro jugador = X, rival al azar = O):")
for k, v in res.items(): print(f"  {k}: {v}")
print(f"\nDerrotas de nuestro jugador: {res['gana O']}.  Jugar perfecto se nota.")


In [ ]:
fig, ax = plt.subplots(figsize=(5.4,3.3))
cols = {"gana X":"#2a9d4a", "empate":"#888", "gana O":"#c0392b"}
barras = ax.bar(list(res.keys()), list(res.values()), color=[cols[k] for k in res])
ax.set_ylabel("partidas (de 2000)")
ax.set_title("Jugador perfecto (X) contra el azar (O)")
for b, v in zip(barras, res.values()):
    ax.text(b.get_x()+b.get_width()/2, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.show()


## 8. ¿Y si empieza el otro? El segundo tampoco pierde

Solemos pensar que quien empieza tiene ventaja. La tiene para *intentar* ganar, pero **no para no
perder**: con juego perfecto, el tres en raya es empate **empiece quien empiece**. Lo comprobamos al
revés: ahora el azar abre (juega `X`) y nuestro jugador perfecto responde de segundo (juega `O`). No
debería perder ni una.


In [ ]:
res2 = juega(azar, mejor_jugada, 2000, semilla=7)
print("Ahora abre el AZAR (X) y nuestro jugador perfecto va de segundo (O):")
for k, v in res2.items(): print(f"  {k}: {v}")
print(f"\nDerrotas de nuestro jugador (que iba segundo): {res2['gana X']}.")
print("Empezar ayuda a ganar, pero no hace falta para no perder.")


## 9. Torneo: perfecto contra perfecto, y contra un rival que falla a veces

Dos jugadores perfectos enfrentados solo pueden empatar: ninguno encuentra una jugada que mejore el
empate. Pero contra un rival que **se equivoca de vez en cuando**, minimax **explota** esos errores y
empieza a ganar. Montamos un rival que juega bien el 80% de las veces y al azar el 20% restante.


In [ ]:
def mixto(t, jugador):
    return mejor_jugada(t, jugador) if random.random() < 0.8 else azar(t, jugador)

torneo_pp = juega(mejor_jugada, mejor_jugada, 50, semilla=3)
print(f"Perfecto (X) contra perfecto (O), 50 partidas: {torneo_pp}")
print("Como ambos juegan optimo y de forma determinista, siempre es el mismo empate.\n")

res3 = juega(mejor_jugada, mixto, 1000, semilla=5)
print("Perfecto (X) contra rival que acierta el 80% y falla el 20% (O), 1000 partidas:")
for k, v in res3.items(): print(f"  {k}: {v}")
print(f"\nGana X: {res3['gana X']}, pierde: {res3['gana O']}. Frente al rival perfecto solo empataba;")
print("en cuanto el rival comete errores, el jugador optimo los castiga.")


## 10. Por qué el ajedrez no se resuelve así: cortar y estimar

Todo lo anterior funciona porque el árbol del tres en raya es pequeño. El del **ajedrez** es
**astronómico** (más posiciones que átomos en el universo observable): ni con poda ni con tablas de
transposición se explora entero. La solución, desde Shannon, es **cortar a cierta profundidad** y, en
vez de llegar al final, **estimar** quién va mejor con una *función de evaluación*. Vamos a verlo en
pequeño con el tres en raya: una evaluación tonta que cuenta **líneas todavía ganables** para `X` menos
las de `O`, y un minimax que se detiene a una profundidad `D` y devuelve esa estimación.


In [ ]:
def lineas_abiertas(t, jug):
    rival = "O" if jug == "X" else "X"
    return sum(1 for a,b,c in GANAN if rival not in (t[a], t[b], t[c]))
def evalua(t):
    # heuristica: lineas que X aun puede completar menos las que aun puede completar O
    return lineas_abiertas(t, "X") - lineas_abiertas(t, "O")

def minimax_d(t, jugador, prof):
    g = ganador(t)
    if g == "X": return 100
    if g == "O": return -100
    if lleno(t): return 0
    if prof == 0: return evalua(t)         # se corta y se ESTIMA
    libres = [i for i in range(9) if not t[i]]
    if jugador == "X":
        return max(minimax_d(t[:i]+["X"]+t[i+1:], "O", prof-1) for i in libres)
    return min(minimax_d(t[:i]+["O"]+t[i+1:], "X", prof-1) for i in libres)

def por_profundidad(D):
    def jugador(t, j):
        sig = "O" if j == "X" else "X"
        libres = [i for i in range(9) if not t[i]]
        punt = [(minimax_d(t[:i]+[j]+t[i+1:], sig, D-1), i) for i in libres]
        return (max if j == "X" else min)(punt)[1]
    return jugador

print("Tablero de ejemplo y su estimacion heuristica (positivo = mejor para X):")
ej = ["X","","O","","X","","","",""]; pinta(ej)
print(f"\nevalua = {evalua(ej)}  (lineas ganables por X: {lineas_abiertas(ej,'X')}, por O: {lineas_abiertas(ej,'O')})")


Ahora la prueba que importa: enfrentamos al azar tres versiones de `X` que solo se diferencian en
**cuánto miran hacia delante** —profundidad 1 (casi a ciegas, solo la jugada inmediata), profundidad 3,
y el jugador **perfecto** (mira hasta el final)—. A más profundidad, debería ganar más y, sobre todo,
perder menos.


In [ ]:
print(f"{'jugador X':<26}{'gana X':>8}{'empate':>9}{'gana O (pierde)':>18}")
for nombre, jx in [("profundidad 1 (a ciegas)", por_profundidad(1)),
                   ("profundidad 3",            por_profundidad(3)),
                   ("perfecto (hasta el final)", mejor_jugada)]:
    r = juega(jx, azar, 400, semilla=2)
    print(f"{nombre:<26}{r['gana X']:>8}{r['empate']:>9}{r['gana O']:>18}")
print("\nMirar poco hace cometer fallos; mirar hasta el final no pierde nunca.")
print("En el ajedrez no se puede mirar hasta el final: por eso se corta a una profundidad y se estima,")
print("y mejor la estimacion, mejor se juega. Cuando ni sabemos disenarla, se APRENDE (refuerzo).")


## 11. Ejemplos cotidianos: pensar lo que hará el otro

Minimax no es solo para juegos de tablero. Su idea —**elige tu mejor opción suponiendo que el otro
elegirá la suya**— aparece en cuanto hay un segundo actor con intereses contrarios:

- **Negociar:** antes de poner una oferta, piensas cómo responderá la otra parte a cada posible
  propuesta, y eliges la propuesta cuya *peor respuesta* te sigue conviniendo. Eso es minimizar el
  máximo daño: minimax puro.
- **Competir en un mercado:** al fijar un precio, anticipas la reacción del competidor; no eliges el
  precio que sería ideal «si el otro no hiciera nada», sino el que aguanta su mejor contraataque.
- **Descartar por sentido común:** cuando dices «esa jugada no la hago, porque me dejaría vendido», estás
  podando una rama sin recorrerla entera. Eso es alfa-beta hecho a mano.

La diferencia entre la intuición y el algoritmo es que el algoritmo **no se cansa ni se despista**: mira
todas las respuestas, no solo las que se le ocurren.


## 12. Pruébalo tú

1. **Juega tú contra él:** escribe un bucle que pida tu casilla con `input()` y deje responder a
   `mejor_jugada`. No vas a ganarle; como mucho, empatar.
2. **Sube la profundidad** del jugador heurístico: `por_profundidad(2)`, `por_profundidad(4)`... ¿a
   partir de qué profundidad deja de perder contra el azar?
3. **Cambia la heurística:** prueba a darle más peso al centro o a las jugadas que crean dos amenazas a
   la vez. ¿Mejora con menos profundidad?
4. **Mide el tamaño de la tabla** de transposición tras una partida real: `len(memo)`. Compáralo con las
   visitas de minimax puro.
5. **Sube a un 4×4** (cuatro en raya): el árbol crece tanto que explorarlo entero deja de ser viable, y
   entiendes en tus carnes por qué hace falta cortar y estimar.


## 13. Errores comunes

- **Olvidar que minimax asume un rival perfecto.** Contra un rival que se equivoca sigue siendo seguro
  (no pierde), pero, como viste en el torneo, hay margen para *explotar* mejor sus errores.
- **Creer que la poda o la memoización cambian la decisión.** No: dan exactamente el mismo valor que
  minimax. Solo cambian cuánto trabajo cuesta.
- **Ignorar el orden de jugadas.** Ordenar bien (probar primero lo prometedor) es lo que hace la poda
  realmente eficaz; con mal orden, apenas poda.
- **Intentar explorar el árbol entero de un juego grande.** Imposible. Hay que cortar a una profundidad
  y estimar con una función de evaluación.
- **Confiar a ciegas en una heurística pobre.** A poca profundidad, una mala estimación hace cometer
  fallos garrafales; gran parte del oficio está en la función de evaluación.


## 14. Qué te llevas

- **Minimax** decide asumiendo que el otro juega lo mejor posible: por eso su jugador no pierde, ni
  abriendo ni respondiendo. El valor `0` del tres en raya demuestra que, bien jugado, no hay victoria
  que forzar.
- Hay tres formas baratas de hacerlo eficiente sin cambiar ni una decisión: **no recalcular** lo ya
  resuelto (tabla de transposición), **podar** lo que ya no puede importar (alfa-beta) y **ordenar** las
  jugadas para podar antes. Las tres son ideas reutilizables en toda la IA clásica.
- Cuando el árbol es gigante (ajedrez, Go) no se explora entero: se **corta y se estima**. La calidad de
  la estimación lo es casi todo... y cuando ni sabemos diseñarla, se **aprende** a base de jugar.

**Para seguir:** el facsímil 10 cambia «explorar el árbol» por «aprender a decidir jugando» (el camino de
AlphaZero); y la idea de modelar al otro actor reaparece en agentes que negocian o compiten (facsímil 5).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*